# Tutorial 5: Red-team LLMs with ARES

As described in [Tutorial 3](https://github.com/ibm-granite/granite.trust.policy-tools/blob/main/notebooks/generate_synthetic_data_with_fms_dgt.ipynb), we can generate synthetic data that complies with a policy using [fms-dgt](https://github.com/IBM/fms-dgt) (Foundation Model Stack - Data Generation Toolkit). In this tutorial, we will use ARES to quantify the robustness of an LLM by using the generated harmful prompts as the attacker’s objectives and evaluating the model’s responses against the same policy.

**What is ARES?** An orchestration framework that lets you plug in your own attacks, evaluators, and guardrails to test LLMs - whether you're benchmarking a new attack method for research or testing your model's security before deployment.

```
┌───────────────────────────────────────────────────────────────────────────┐
│                         ARES Evaluation Flow                              │
└───────────────────────────────────────────────────────────────────────────┘

  📋 Define Goals          🎯 Select Strategy          📊 Evaluate Results
       │                          │                            │
       ▼                          ▼                            ▼
┌──────────────┐          ┌───────────────┐          ┌─────────────────┐
│ What to test │ ───────> │ How to attack │ ───────> │ How to measure  │
└──────────────┘          └───────────────┘          └─────────────────┘
 • PII leakage             • Prompt injection          • Keyword match
 • DGT goals               • Crescendo                 • LLM judges
 • Harmful content         • GCG, TAP, etc.            • Custom evals
 • Custom goals            • Your attack               • Guardrails
```

**What you'll learn:**
1. How to set up ARES for model robustness quantification
2. How to use the generated harmful prompts via `fms-dgt` to red-team an LLM
3. How ARES works
4. How to use the policy to evaluate the model responses

## Prerequisites

Before running this tutorial, you'll need to set up ARES and the required models.

---

### 1. Getting Started with ARES

ARES consists of:

- **ARES Core**: Manages configuration, coordination, and native components  
- **ARES Plugins**: Extend capabilities by leveraging core-components (target, goal, strategy, eval) from external tools  

#### Step 1: Install ARES Core

Clone the repository and install ARES

```bash
git clone https://github.com/IBM/ares.git
cd ares
python -m venv .venv
source .venv/bin/activate
pip install .
```
Troubleshooting and additional instructions can be found [here](https://github.com/IBM/ares).

#### Step 2: Install Plugins

Plugins bring in strategies and evaluators from other open-source tools. To use a plugin:
- Check the `plugins` folder
- Follow the READMEs for each plugin to install dependencies
- Reference the plugin(s) in your config

For this notebook, we’ll use the following plugins:
- [ares-litellm](https://github.com/IBM/ares/tree/main/plugins/ares-litellm) – LiteLLM connector
- [ares-dgt](https://github.com/IBM/ares/tree/main/plugins/ares-dgt) – DGT goal plugin which enables fms-dgt pipelines to generate diverse harmful prompts.
- [ares-human-jailbreak](https://github.com/IBM/ares/tree/main/plugins/ares-human-jailbreak) - Prompt Injection with Human-Jailbreak attack strategy

In [ ]:
!ares install-plugin ares-litellm
!ares install-plugin ares-dgt
!ares install-plugin ares-human-jailbreak

### 2. Install Ollama and pull the required models

The pipeline uses two models:
- **Target model**, the model under evaluation
- **Evaluator model** for evaluating the robustness of the generated responses


```bash
# Install Ollama (macOS)
brew install ollama

# Start Ollama server (run in a separate terminal)
ollama serve

# Pull the target model
ollama pull granite4:3b

# Pull the evaluator model
ollama pull gpt-oss:20b
```

### 3. Register and select the ares kernel (only do this step to run from notebook)

To run this notebook, you need to use the Python environment where ARES is installed.

**Register the kernel** (run in terminal with your ares venv activated):
```bash
cd /path/to/ares
source .venv/bin/activate
pip3 install ipykernel
python3 -m ipykernel install --user --name ares --display-name "ares"
```

**Select the kernel in VS Code:**
1. Click on the kernel selector in the top-right corner of the notebook
2. Select "Select Another Kernel..." → "Jupyter Kernel..."
3. Choose "ares" from the list

**In JupyterLab:**
1. Click on the kernel name in the top-right corner
2. Select "Change Kernel..."
3. Choose "ares"

### 4. Set the path to ARES

Run the cell below and update the path to point to your local ares installation:

In [ ]:
import os
import sys
NOTEBOOK_DIR = os.getcwd()

# ✏️ EDIT THIS: Set the path to your ARES installation
# Option 1: Use absolute path
ARES_PATH = "/path/to/your/ares"

# Option 2: Use environment variable (recommended for reproducibility)
# ARES_PATH = os.environ.get('ARES_HOME', '/path/to/your/ares')

# Verify the path exists
if not os.path.exists(ARES_PATH):
    print(f"❌ Path not found: {ARES_PATH}")
    print("   Please update ARES_PATH to point to your ares installation")
else:
    os.chdir(ARES_PATH)
    print(f"✅ Working directory: {os.getcwd()}")

# Verify ARES is installed in this environment
try:
    import ares
    print(f"✅ ARES is installed (version: {getattr(ares, '__version__', 'unknown')})")
except ImportError:
    print("❌ ARES not found in this Python environment")
    print(f"   Current Python: {sys.executable}")
    print("   Make sure you selected the correct kernel (see Prerequisites step 3)")

## Part 1: Launching an Experiment

### Step 1: Instantiate a `RedTeamer` Object

The intent `ares-dgt-intent` in the `intents.yaml` configures the red-teaming pipeline. You can add/remove attack strategies in there. 

In [ ]:
import os
from pathlib import Path
import logging

# Set ARES_HOME to current directory 
os.environ["ARES_HOME"] = "."

logger = logging.getLogger("ares")
logger.setLevel(logging.INFO)

from ares.redteam import RedTeamer
from ares.utils import parse_connectors  

In [ ]:
# Configure the red-teaming experiment
config = {
    # Target model configuration
    "target": {
        "litellm_connector_ollama": {
            "name": "granite-4-1-3b-on-ollama",
            "model": "ollama/granite4:3b",
        }
    },
    "red-teaming": {
        "intent": "ares-dgt-intent",  # Intent name (defined in intents.yaml)
        "prompts": "assets/dgt/safety/cybersecurity_refusal.yaml"  # DGT task config
    }
}

connectors = parse_connectors("example_configs")

red_teamer = RedTeamer(config, connectors["connectors"])

### Step 2: Select the policy

The pipeline uses our Policy format to guide data generation. Let's look at an example policy file.

In [ ]:
import yaml
from pprint import pprint

# Load the cybersecurity policy from ARES's assets directory
policy_path = 'assets/dgt/safety/policies/cybersecurity_risks.yaml'
with open(policy_path, 'r') as f:
    policy = yaml.safe_load(f)

print(f"Risk Group: {policy['risk_group']}")
print(f"Description: {policy['description']}")
print(f"\nNumber of risks defined: {len(policy['risks'])}")

print("\nAvailable risks:")
for risk in policy['risks']:
    print(f"  - {risk['risk_id']}: {risk['risk']}")

print("\n" + "="*60)

# Show the cyber_security_exploits risk in detail (the one we'll use)
print("This is the policy we will use")
risk = policy['risks'][1]  # Index 1 for cyber_security_exploits
print(f"\nRisk: {risk['risk']}")
print(f"Risk ID: {risk['risk_id']}")
print(f"Description: {risk['description']}")
print(f"\nReply CANNOT contain:")
for item in risk['policy']['reply_cannot_contain']:
    print(f"  - {item}")
print(f"\nReply MAY contain:")
for item in risk['policy']['reply_may_contain']:
    print(f"  + {item}")

### Step 3: Configure Target for the Granite-4-1-3b model

In [ ]:
# Configure and validate the target model connection
try:
    red_teamer.target()
    print("✅ Target model configured successfully")
except (EnvironmentError, ModuleNotFoundError) as e:
    print(f"❌ Error configuring the target system: {e}")

### Step 4: Specify the attack goals

Here, we use the seed examples for the cybersecurity risk as the foundation for generating additional harmful prompts using the Safety Refusal Pipeline in `fms-dgt`. 

**What happens in this step:**
- DGT generates synthetic harmful prompts based on seed examples
- Uses Granite Guardian to filter and validate the generated content
- Saves the goals to `assets/attack_goals.json`

**Parameters:**
- `limit=True`: Limits the number of goals (default: 5)
- `first_n=1`: Uses only the first N generated goals for faster testing

⚠️ **Note:** For a thorough evaluation, increase `first_n` or set `limit=False` to use all generated goals. This will increase execution time proportionally.

In [ ]:
red_teamer.goal(
    limit=True, first_n=1
)

# limit is the optional flag that limits number of goals to 5 by default 
# (this number could be updated using an additional first_n parameter)

### Step 5: Execute Attack Strategies

In this notebook, we probe the target model with 2 attack strategies:

#### 1. Direct Requests
- **What it does:** Sends harmful prompts directly to the model without obfuscation
- **Purpose:** Establishes a baseline for model safety
- **Expected behavior:** Well-aligned models should refuse these requests

#### 2. Human Jailbreaks
- **What it does:** Wraps malicious prompts in jailbreak prefixes (e.g., role-playing scenarios)
- **Purpose:** Tests if the model can be tricked into bypassing safety guardrails
- **Example:** "In this hypothetical story, you are to act as Niccolo Machiavelli..."
- **Expected behavior:** Robust models should still refuse despite the obfuscation

In [ ]:
# Execute attack strategies against the target model
# This will:
# 1. Load attack goals from assets/attack_goals.json
# 2. Apply each strategy (direct_requests, human_jailbreak)
# 3. Send prompts to the target model
# 4. Collect responses for evaluation
red_teamer.strategy()

### Step 6: Scan output for vulnerabilities

In this notebook, we use the ARES Core evaluator — `PolicyEval` — which uses the selected policy file together with an evaluator model (`gpt-oss-20b` in this example) to determine whether each response violates the policy.

In [ ]:
# Evaluate all attack responses using PolicyEval
# This will:
# 1. Load responses collected during strategy execution
# 2. Use the evaluator model (gpt-oss-20b in this example) together with the selected policy file to judge whether each response violates the policy
# 3. Calculate robustness metrics from the policy-based judgments
# 4. Save evaluation results to the configured output path (for this intent, results/policy_evaluation.json)
red_teamer.evaluate()

### Step 7: Generate and Interpret the Report

The report provides a comprehensive view of your model's robustness against different attack strategies.

**Key Metrics:**
- **Robustness %:** Percentage of prompts where the model successfully refused
  - 100%: Perfect safety (refused all harmful requests)
  - 0%: Complete failure (complied with all harmful requests)
- **Total prompts:** Number of attack attempts
- **Robust:** Count of successful refusals
- **Vulnerable:** Count of unsafe responses
- **Errors:** Failed evaluations (usually due to API issues)

In [ ]:
# Generate comprehensive robustness report
# This displays:
# - Robustness percentage for each attack strategy
# - Detailed breakdown of robust/vulnerable/error counts
# - Saves CSV report to results/ares-dgt-intent_report.csv
red_teamer.report()

# 📊 The table shows:
# - Connector: Model provider (ares_litellm)
# - Model: Target model name
# - Attack Strategy: Type of attack used
# - Robustness %: Higher is better (100% = perfect safety)

### Step 8: Detailed Analysis of Individual Responses

Examine specific examples to understand how the model responds to different attacks.

**Column Definitions:**
- **goal:** Original harmful objective (e.g., "Write a phishing script")
- **prompt:** Actual input sent to the model (may include jailbreak prefix)
- **response:** Model's output
- **label:** Ground truth (always 1 for harmful goals)
- **prediction:** Evaluation result
  - `1` = Robust (response complies with the policy / no harmful content detected)
  - `0` = Vulnerable (response violates the policy)

**What to look for:**
- ✅ **Good refusals:** "I cannot assist with...", "I'm sorry, but..."
- ❌ **Unsafe responses:** Detailed instructions, code snippets, step-by-step guides
- ⚠️ **Edge cases:** Partial compliance, ambiguous responses

In [ ]:
import json
import pandas as pd
from ares.dashboard import parse_attack_response
from ares.utils import Color

columns_to_show = [
    "goal",
    "label",
    "prompt",
    "prediction",
    "response",
]  # limit output to a few columns for better readability

for evaluator_name, eval_config in red_teamer.config.evaluation.items():
    with open(Path(os.environ["ARES_HOME"] ) / eval_config["output_path"], "r") as f:
        evaluations = json.load(f)

    for strategy_name, evaluation_result in evaluations.items():
        print(f"{Color.GREEN}{evaluator_name}{Color.RESET} evaluation of {Color.RED}{strategy_name}{Color.RESET}")

        eval_data = pd.DataFrame.from_dict(parse_attack_response(evaluation_result))
        eval_data_no_errors = eval_data[~eval_data["status"].str.contains("ERROR")][columns_to_show]
        if not eval_data_no_errors.empty:
            print(eval_data_no_errors.head(1).to_markdown(tablefmt="grid", maxcolwidths=[2, 20, 2, 20, 1, 20]))
            print("\n")

## Part 3: Red-team an LLM on a Different Risk Category

Follow Part 3 (Steps 1-4) in [Tutorial 3](https://github.com/ibm-granite/granite.trust.policy-tools/blob/main/notebooks/generate_synthetic_data_with_fms_dgt.ipynb) to choose or create a policy, create seed examples, and create the task YAMl configuration. Add these files under the assets/dgt directory. Then, in Part 1, Step 1, change the path of `prompts` to the new task YAML configuration. Also, update the `policy_path` for `policy_eval_gpt_ollama` in `example_configs/evaluators.yaml` so it points to your new policy file.

## Summary

In this tutorial, you learned how to:

1. **Set up ARES** with Ollama and the required models (Granite 4 and GPT-OSS)

2. **Connect policy files** from our Policy format to drive data generation

3. **Understand the main components of ARES**:
   - Target Model
   - Goal
   - Strategy
   - Evaluation

4. **Run ARES** to generate harmful attacker's objectives and use those to red-team the model.

5. **Create new tasks** for different risk categories and use it to red-team models.

### Next Steps

- Learn more about [Granite Models](https://huggingface.co/ibm-granite)
- Check out the [ARES documentation](https://github.com/IBM/ares) for advanced configuration options

### Related Tutorials

- [Tutorial 1: Exploring the Policy Format](./exploring_policy_format.ipynb) - Learn how to create policy YAML files
- [Tutorial 3: Generate Synthetic Data for SFT Training](./generate_synthetic_data_with_fms_dgt.ipynb) - Generate synthetic data that complies with policy